# Fed-KAN on Colab GPU

FedAvg across LargeST districts, FC head vs Taylor-KAN head.

Before running: Runtime > Change runtime type > T4 GPU. Expects this in Google Drive:

```
MyDrive/fedkan/
  fedkan_code.zip
  data_real/district_3/ ...
```

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# run settings
ROUNDS = 10
RESAMPLE = 3        # 3 = 15-min data, same as Phase 1
BATCH = 64
MAX_BATCHES = 0     # 0 = full epoch per client
TAYLOR_ORDER = 2

DRIVE = '/content/drive/MyDrive/fedkan'
WORK = '/content/fedkan'

In [ ]:
# copy code and data off Drive onto the local disk (much faster to read)
import os, shutil, zipfile
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
with zipfile.ZipFile(f'{DRIVE}/fedkan_code.zip') as z:
    z.extractall(WORK)
shutil.copytree(f'{DRIVE}/data_real', f'{WORK}/phase2/data_real')
print(sorted(os.listdir(f'{WORK}/phase2/data_real')))

In [ ]:
%cd /content/fedkan/phase2
COMMON = f'--data data_real --rounds {ROUNDS} --resample {RESAMPLE} --batch-size {BATCH} --max-batches {MAX_BATCHES} --taylor-order {TAYLOR_ORDER}'

In [ ]:
!python run_federated_local.py --head fc {COMMON} --out results/colab_fc.json

In [ ]:
!python run_federated_local.py --head taylor {COMMON} --out results/colab_taylor.json

In [ ]:
# compare the two runs and save everything back to Drive
import json
import matplotlib.pyplot as plt

runs = {}
for head in ['fc', 'taylor']:
    with open(f'results/colab_{head}.json') as f:
        runs[head] = json.load(f)

print(f"{'head':8}{'params':>10}{'MAE':>9}{'RMSE':>9}{'R2':>8}")
for head, r in runs.items():
    last = r['rounds'][-1]
    print(f"{head:8}{r['head_params']:>10,}{last['mae']:>9.3f}{last['rmse']:>9.3f}{last['r2']:>8.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for head, r in runs.items():
    rounds = [x['round'] for x in r['rounds']]
    ax[0].plot(rounds, [x['mae'] for x in r['rounds']], marker='o', label=head)
    ax[1].plot(rounds, [x['r2'] for x in r['rounds']], marker='o', label=head)
ax[0].set_title('Validation MAE')
ax[1].set_title('Validation R2')
for a in ax:
    a.set_xlabel('round')
    a.legend()
plt.tight_layout()
plt.savefig('results/colab_fc_vs_taylor.png', dpi=150)
plt.show()

os.makedirs(f'{DRIVE}/results', exist_ok=True)
for name in ['colab_fc.json', 'colab_taylor.json', 'colab_fc_vs_taylor.png']:
    shutil.copy(f'results/{name}', f'{DRIVE}/results/{name}')
print('saved to', f'{DRIVE}/results')